# Limpar Landing Zone — Orquestrador de Manutenção

Ponto de entrada único que remove partições `data=AAAA-MM-DD/` mais antigas que a retenção definida (padrão 30 dias, ADR-009), por sistema.

Modo `dry_run` (padrão `true`) só lista o que seria removido, sem remover de fato — validação antes de executar algo irreversível.

`data_referencia` é parametrizável para permitir testar sem esperar 30 dias reais passarem, usando uma data simulada no futuro.

Cada execução é registrada em `observability.pipeline_runs` (ADR-014).

Referências: ADR-008 (Widgets), ADR-009 (retenção da Landing Zone), ADR-014.

## Widgets

- `data_referencia`: data de referência para o cálculo de retenção (AAAA-MM-DD). Vazio = hoje real.
- `dias_retencao`: quantos dias manter na Landing Zone antes de remover. Padrão 30.
- `dry_run`: `"true"` (padrão, só lista) ou `"false"` (remove de fato).
- `sistema`: `"todos"` ou um sistema específico.

In [0]:
dbutils.widgets.text("data_referencia", "", "Data de referência (AAAA-MM-DD)")
dbutils.widgets.text("dias_retencao", "30", "Dias de retenção")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "Dry run")
dbutils.widgets.dropdown("sistema", "todos", ["todos", "crm", "erp", "tms", "financeiro"], "Sistema")

In [0]:
from datetime import date

data_referencia_str = dbutils.widgets.get("data_referencia")
data_referencia = date.fromisoformat(data_referencia_str) if data_referencia_str else date.today()

dias_retencao = int(dbutils.widgets.get("dias_retencao"))
dry_run = dbutils.widgets.get("dry_run") == "true"
sistema_selecionado = dbutils.widgets.get("sistema")

print(f"data_referencia: {data_referencia}")
print(f"dias_retencao: {dias_retencao}")
print(f"dry_run: {dry_run}")
print(f"sistema: {sistema_selecionado}")

## Execução

Remove (ou lista, em dry_run) partições vencidas de cada sistema selecionado. Cada execução é registrada em `observability.pipeline_runs`.

In [0]:
from src.manutencao.limpar_landing_zone import limpar_landing_zone
from src.observabilidade.registrar_execucao import registrar_execucao_pipeline

sistemas = ["crm", "erp", "tms", "financeiro"] if sistema_selecionado == "todos" else [sistema_selecionado]

resultados = []
for sistema in sistemas:
    resultado = limpar_landing_zone(
        spark=spark, dbutils=dbutils, sistema=sistema,
        dias_retencao=dias_retencao, data_referencia=data_referencia, dry_run=dry_run,
    )
    resultados.append(resultado)
    print(resultado)

    registrar_execucao_pipeline(
        spark=spark, pipeline="limpar_landing_zone", item=sistema,
        status="dry_run" if dry_run else "sucesso", detalhes=resultado,
    )

print("\nResumo:")
for r in resultados:
    print(f"  {r['sistema']}: {len(r['particoes_a_remover'])} partição(ões) a remover")

In [0]:
df_runs = spark.table("poc_pulse_observability.observability.pipeline_runs")
df_runs.filter(df_runs.pipeline == "limpar_landing_zone").show(10, truncate=False)